# UEBA Portable — Entraîner sur le **normal**, tester sur les **attaques**

Démarche en deux temps, comme en production :

1. **Apprentissage** : on entraîne un modèle par utilisateur sur un dataset
   **normal** (sans incident) — il apprend la baseline de chacun.
2. **Test** : on charge un **second** dataset **contenant les attaques** et on
   regarde ce que le modèle détecte → **Recall, Précision, F1, matrice de
   confusion, Recall opérationnel**.

> Tu uploades **deux** fichiers : (1) le normal, (2) celui avec les attaques.

## 0. Installation

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q --upgrade --force-reinstall --no-deps "git+https://github.com/assia-xnz/ueba-portable.git"
    print("✅ Installé. Si des cellules ont déjà tourné : Exécution → Redémarrer la session.")
else:
    sys.path.insert(0, "../src")
print("Environnement :", "Google Colab" if IN_COLAB else "Local")

## 1. Upload du dataset **NORMAL** (entraînement)

Le fichier **sans attaque**, qui définit le comportement de référence.

In [ ]:
if IN_COLAB:
    from google.colab import files

    print("📂 Choisis le fichier NORMAL (entraînement) :")
    NORMAL_CSV = next(iter(files.upload()))
else:
    NORMAL_CSV = "../tests/integration/fixtures/sample_logs.csv"
print("Dataset normal :", NORMAL_CSV)

## 2. Entraînement de la baseline sur le normal

Un modèle dédié par utilisateur, sur 100 % du normal (`train_ratio=1.0`),
avec `svm_nu=0.05` pour limiter les faux positifs.

In [ ]:
import csv
from datetime import timedelta
from pathlib import Path

from ueba.adapters.wazuh import WazuhAdapter
from ueba.domain.features import UEBAFeatureExtractor
from ueba.domain.per_user_ensemble import PerUserAnomalyEnsemble

MIN_WINDOWS = 30  # baisse (ex. 15/10) si des utilisateurs ont peu de fenêtres


def load_vectors(path):
    with Path(path).open(newline="", encoding="utf-8") as f:
        records = list(csv.DictReader(f))
    events = WazuhAdapter().normalize(records)
    extractor = UEBAFeatureExtractor(
        window_size=timedelta(hours=1), window_step=timedelta(minutes=30)
    )
    return events, extractor.extract(events)


_, train_vectors = load_vectors(NORMAL_CSV)
print(f"Fenêtres d'apprentissage (normal) : {len(train_vectors)}")

model = PerUserAnomalyEnsemble(
    min_windows_per_user=MIN_WINDOWS,
    train_ratio=1.0,
    svm_nu=0.05,
    n_estimators=200,
    majority_threshold=2,
    random_state=42,
)
model.fit(train_vectors)
print(f"Modèles entraînés : {len(model.trained_users)}")
print("Utilisateurs avec baseline :", model.trained_users)

## 3. Upload du dataset de **TEST** (avec les attaques)

Le fichier qui **contient les attaques** à détecter.

In [ ]:
if IN_COLAB:
    print("📂 Choisis le fichier de TEST (avec attaques) :")
    TEST_CSV = next(iter(files.upload()))
else:
    TEST_CSV = "../tests/integration/fixtures/sample_logs.csv"
print("Dataset de test :", TEST_CSV)

## 4. Vérité terrain (dans le fichier de test)

Jours et comptes ciblés par l'attaque connue dans **ce** fichier de test.

In [ ]:
ATTACK_DATES = ["2026-05-13", "2026-05-16"]
ATTACK_USERS = {
    "a.amrani",
    "l.idrissi",
    "l.mus",
    "y.ben",
    "n.alam",
    "s.ed",
    "k.alaa",
}
print("Jours d'attaque :", ATTACK_DATES)
print("Comptes ciblés  :", sorted(ATTACK_USERS))

## 5. Test : détection sur le fichier d'attaque + métriques

On applique le modèle (appris sur le normal) au fichier de test. Une *fenêtre
d'attaque* = (utilisateur ciblé) ET (jour d'attaque).

In [ ]:
import pandas as pd

_, test_vectors = load_vectors(TEST_CSV)
verdicts = model.predict(test_vectors)

df = pd.DataFrame(
    {
        "user": [v.user for v in test_vectors],
        "date": [v.window_start.date().isoformat() for v in test_vectors],
        "window_start": [v.window_start for v in test_vectors],
        "is_anomaly": [d.is_anomaly for d in verdicts],
        "was_in_training": [d.was_in_training for d in verdicts],
    }
)
df["is_attack"] = df["user"].isin(ATTACK_USERS) & df["date"].isin(ATTACK_DATES)

untrained = df[(~df["is_attack"]) & (~df["was_in_training"])]["user"].nunique()
if untrained:
    print(f"⚠️  {untrained} utilisateur(s) normal(aux) sans modèle (default-deny) "
          f"→ baisse MIN_WINDOWS (cellule 2) pour un FP juste.\n")

TP = int((df["is_attack"] & df["is_anomaly"]).sum())
FN = int((df["is_attack"] & ~df["is_anomaly"]).sum())
FP = int((~df["is_attack"] & df["is_anomaly"]).sum())
TN = int((~df["is_attack"] & ~df["is_anomaly"]).sum())

recall = TP / (TP + FN) if (TP + FN) else float("nan")
precision = TP / (TP + FP) if (TP + FP) else float("nan")
f1 = 2 * precision * recall / (precision + recall) if precision and recall else float("nan")
fp_rate = FP / (FP + TN) if (FP + TN) else float("nan")

attack = df[df["is_attack"]]
op_recall = attack.groupby(["user", "date"])["is_anomaly"].any().mean() if len(attack) else float("nan")
first = attack.sort_values("window_start").groupby(["user", "date"]).first()["is_anomaly"]

print("===== MATRICE DE CONFUSION (fenêtres) =====")
print(f"  TP={TP:5d}   FN={FN:5d}")
print(f"  FP={FP:5d}   TN={TN:5d}\n")
print("===== MÉTRIQUES =====")
print(f"  Recall (fenêtre)      : {recall:6.1%}")
print(f"  Précision             : {precision:6.1%}")
print(f"  F1-score              : {f1:6.1%}")
print(f"  FP rate               : {fp_rate:6.1%}")
print(f"  Recall opérationnel   : {op_recall:6.1%}  (user×jour avec ≥1 alerte)")
print(f"  Détection 1re fenêtre : {int(first.sum())}/{len(first)}")

## 6. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (axc, axt) = plt.subplots(1, 2, figsize=(14, 5))

mat = np.array([[TP, FN], [FP, TN]])
axc.imshow(mat, cmap="Blues")
axc.set_xticks([0, 1]); axc.set_xticklabels(["prédit\nanomalie", "prédit\nnormal"])
axc.set_yticks([0, 1]); axc.set_yticklabels(["réel\nattaque", "réel\nnormal"])
for i in range(2):
    for j in range(2):
        axc.text(j, i, mat[i, j], ha="center", va="center",
                 color="white" if mat[i, j] > mat.max() / 2 else "black", fontsize=14)
axc.set_title("Matrice de confusion")

normal = df[~df["is_anomaly"]]
flagged = df[df["is_anomaly"]]
axt.scatter(normal["window_start"], normal["user"], s=10, c="#aec7e8", label="normal")
axt.scatter(flagged["window_start"], flagged["user"], s=22, c="#d62728", label="alerte")
for d in ATTACK_DATES:
    axt.axvspan(np.datetime64(f"{d}T00:00:00"), np.datetime64(f"{d}T23:59:59"),
                color="orange", alpha=0.12)
axt.set_title("Timeline (zones orange = jours d'attaque)")
axt.legend(loc="upper right"); axt.tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()

## 7. Lecture

* **Recall opérationnel** (≥ 1 alerte par user×jour d'attaque) = la métrique
  SOC de référence : vise 100 %.
* **FP rate / Précision** = le bruit. Se règle via `svm_nu`,
  `majority_threshold`, `reconstruction_error_percentile`, et `MIN_WINDOWS`
  (pour éviter le default-deny). Documente le compromis Recall ↔ FP dans le
  rapport.